In [ ]:
# Ultralytics (YOLOv8)
!pip install ultralytics --quiet

# OpenCV for image processing
!pip install opencv-python --quiet

# Matplotlib for visualization
!pip install matplotlib --quiet

# (Optional but useful)
!pip install seaborn pandas --quiet


!pip install roboflow --quiet


!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
!pip install opencv-python


Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-python-headless 4.10.0.84
Uninstalling opencv-python-headless-4.10.0.84:
  Successfully uninstalled opencv-python-headless-4.10.0.84
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (72.9 MB)


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="ZeJKDEWv7ZknZBJqtk1s")
project = rf.workspace("ibtihals-workspace").project("sids-mdtwn")
version = project.version(2)
dataset = version.download("yolov8")

from ultralytics import YOLO
import cv2
import os

# Load both models
body_model = YOLO("body_best.pt")  # detects: back, stomach
face_model = YOLO("face_best.pt")  # detects: head, nose

# Set up folders
test_folder = "/content/sids-2/test/images"
output_folder = "results"
os.makedirs(output_folder, exist_ok=True)

results_summary = []

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)
    frame = cv2.imread(img_path)

    #Check if image loaded successfully
    if frame is None:
        print(f"Warning: Could not read {img_name}, skipping.")
        continue

    #Run both models on the frame
    body_results = body_model(frame, verbose=False)[0]
    face_results = face_model(frame, verbose=False)[0]

    #Combine all detections from both models into one list
    all_boxes = []
    for res in [body_results, face_results]:
        for box in res.boxes:
            all_boxes.append({
                "label": res.names[int(box.cls[0])],
                "conf": float(box.conf[0])
            })

    #Extract baby position and face visibility
    position = "none"
    best_pos_conf = 0
    nose_detected = False

    for item in all_boxes:
        lbl = item["label"].lower()
        cnf = item["conf"]

        # Track the highest confidence position (back or stomach)
        if ("back" in lbl or "stomach" in lbl) and cnf > 0.2:
            if cnf > best_pos_conf:
                best_pos_conf = cnf
                position = lbl

        # Check if nose is visible
        if "nose" in lbl and cnf > 0.3:
            nose_detected = True

    #Safety logic
    # Stomach = immediate danger regardless of face visibility
    if "stomach" in position:
        status = "DANGER: STOMACH SLEEPING"
        color = (0, 0, 255)          # Red

    #Back = check face visibility
    elif "back" in position:
        if nose_detected:
            # Back + nose visible = fully safe
            status = "SAFE"
            color = (0, 255, 0)      # Green
        else:
            # Back + nose not visible = face may be covered
            status = "WARNING: FACE MAY BE COVERED"
            color = (0, 165, 255)    # Orange

    # No clear position detected
    else:
        status = "SCANNING..."
        color = (200, 200, 200)      # Gray

    #Save result to summary list
    results_summary.append({
        "image": img_name,
        "position": position,
        "nose": nose_detected,
        "status": status
    })

    #Draw results on the image
    annotated = frame.copy()

    #Body boxes — green
    for box in body_results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = f"{body_model.names[int(box.cls[0])]} {float(box.conf[0]):.2f}"
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(annotated, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    #Face boxes — blue
    for box in face_results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = f"{face_model.names[int(box.cls[0])]} {float(box.conf[0]):.2f}"
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(annotated, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    #Draw status bar at the top
    cv2.rectangle(annotated, (0, 0), (frame.shape[1], 50), (0, 0, 0), -1)
    cv2.putText(annotated, status, (10, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    #Save annotated image
    cv2.imwrite(os.path.join(output_folder, img_name), annotated)

print("Inference done!")

loading Roboflow workspace...
loading Roboflow project...
Inference done!


In [ ]:
# trying the model on real videos

import cv2
from ultralytics import YOLO
import os

#Load both models
print("Loading models ...")
body_model = YOLO("body_best.pt")  # detects: back, stomach
face_model = YOLO("face_best.pt")  # detects: head, nose

#Video settings
video_input = "/content/baby3.MP4"
cap = cv2.VideoCapture(video_input)

if not cap.isOpened():
    print("Error: The video file was not found. Please check the path.")
else:
    fps    = int(cap.get(cv2.CAP_PROP_FPS))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path = "Baby_Guard_Final_Result.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))


    # If baby stays on stomach for 5 seconds straight -> DANGER
    danger_counter    = 0
    WAIT_SECONDS      = 5
    frames_threshold  = fps * WAIT_SECONDS

    print(f"Processing has begun... The video will be saved as: {output_path}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        #Run both models on the current frame
        body_results = body_model(frame, verbose=False, conf=0.25)[0]
        face_results = face_model(frame, verbose=False, conf=0.25)[0]

        #Collect all detected labels from both models
        all_labels = []
        nose_found = False

        for res in [body_results, face_results]:
            for box in res.boxes:
                lbl = res.names[int(box.cls[0])].lower()
                all_labels.append(lbl)
                if "nose" in lbl:
                    nose_found = True

        combined_labels = " ".join(all_labels)


        # Stomach = immediate danger regardless of face visibility
        if "stomach" in combined_labels:
            danger_counter += 1
            if danger_counter < frames_threshold:
                remaining = WAIT_SECONDS - (danger_counter // fps)
                status = f"WARNING: STOMACH ({remaining}s to DANGER)"
                color  = (0, 165, 255)
            else:
                status = "DANGER: STOMACH SLEEPING"
                color  = (0, 0, 255)

        # Back = check face visibility
        elif "back" in combined_labels:
            danger_counter = 0
            if nose_found:
                # Back + nose visible = fully safe
                status = "SAFE"
                color  = (0, 255, 0)     # Green
            else:
                # Back + nose not visible = face may be covered
                status = "WARNING: FACE MAY BE COVERED"
                color  = (0, 165, 255)   # Orange

        # No clear position detected
        else:
            danger_counter = 0
            status = "SCANNING..."
            color  = (200, 200, 200)     # Gray

        #Draw bounding boxes on the frame
        annotated_frame = frame.copy()

        #Body boxes — green
        for box in body_results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            lbl = body_model.names[int(box.cls[0])]
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(annotated_frame, lbl, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        #Face boxes — blue
        for box in face_results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            lbl = face_model.names[int(box.cls[0])]
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(annotated_frame, lbl, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

        #Draw status bar at the top
        cv2.rectangle(annotated_frame, (0, 0), (width, 90), (0, 0, 0), -1)
        cv2.putText(annotated_frame, status, (20, 62),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.4, color, 3)

        out.write(annotated_frame)

    cap.release()
    out.release()
    print(f"Processed successfully! File located here: {output_path}")

Loading models ...
Processing has begun... The video will be saved as: Baby_Guard_Final_Result.mp4
Processed successfully! File located here: Baby_Guard_Final_Result.mp4
